In [14]:
!pip install noisereduce


In [15]:
# pure FFT / IFFT spectral subtraction implementation, no noisereduce,
# no ML, no fancy DSP libraries.
# This is classic textbook spectral noise reduction,
# and it’s exactly what you’d later port to C / embedded / ESP32.

#What this code does
#	1.	Load WAV audio
#	2.	Add white noise
#	3.	Estimate noise spectrum (FFT only)
#	4.	Perform spectral subtraction frame-by-frame
#	5.	Reconstruct signal with IFFT + overlap-add
#	6.	Play:
#	•	original
#	•	noisy
#	•	denoised


# Required packages
# pip install numpy scipy soundfile sounddevice

import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import requests
from scipy.io import wavfile
import io
import requests
import librosa
import noisereduce as nr


In [16]:
# 1. Define the RAW GitHub URL   RAW!!!!!!!!!!  RAW!!!!!!
# The original URL pointed to the HTML preview; this one points to the raw data.
url = 'https://raw.githubusercontent.com/rfurch/unlpam_noise/refs/heads/main/data/originalAudioFile.mp3'

file_name = "originalAudioFile.mp3"

r = requests.get(url)
with open(file_name, "wb") as f:
    f.write(r.content)

print("Downloaded:", file_name)

Downloaded: originalAudioFile.mp3


In [17]:

y, sr = librosa.load("originalAudioFile.mp3", sr=None)

# Display the audio player (optional)
print(f"Waveform shape: {y.shape}")
print(f"Sampling rate: {sr} Hz")

# amplification
y = y * 10

Waveform shape: (375552,)
Sampling rate: 44100 Hz


In [18]:
Audio(y, rate=sr, autoplay=True, normalize=False)


In [19]:
# ----------------------------
# Add white noise
# ----------------------------
noise_level = 0.02
y_noisy = y + noise_level * np.random.randn(len(y))

In [20]:
Audio(y_noisy, rate=sr, autoplay=True, normalize=False)


In [21]:
# ----------------------------
# Noise profile
# Use first 0.5 seconds as "noise only"
# ----------------------------
noise_sample_duration = 0.9  # seconds
noise_samples = int(noise_sample_duration * sr)
noise_clip = y_noisy[:noise_samples]


In [22]:

# ----------------------------
# Spectral noise reduction
# The noisereduce library:
#   •	Performs STFT
#	•	Estimates noise spectrum
#	•	Attenuates noise bins
#	•	Reconstructs the signal via inverse STFT
# This is true spectral subtraction, not just filtering.
# If noise is constant use a longer noise-only segment:
# noise_sample_duration = 1.0
# ----------------------------

output = nr.reduce_noise(
    y=y_noisy,
    y_noise=noise_clip,
    sr=sr,
    prop_decrease=1.0
)
# Tune parameters  prop_decrease=0.8   # softer noise removal


In [23]:
Audio(output, rate=sr, autoplay=True)
